# Loan Default – Preprocessing Pipeline
Dataset: [nikhil1e9/loan-default](https://www.kaggle.com/datasets/nikhil1e9/loan-default)

| Column | Type | Description |
|---|---|---|
| LoanID | str | Unique identifier – will be dropped |
| Age | int | Borrower age |
| Income | float | Annual income |
| LoanAmount | float | Requested loan amount |
| CreditScore | int | Credit score (300–850) |
| MonthsEmployed | int | Months at current employer |
| NumCreditLines | int | Number of open credit lines |
| InterestRate | float | Loan interest rate (%) |
| LoanTerm | int | Loan duration in months |
| DTIRatio | float | Debt-to-income ratio |
| Education | str | High School / Bachelor's / Master's / PhD |
| EmploymentType | str | Full-time / Part-time / Self-employed / Unemployed |
| MaritalStatus | str | Single / Married / Divorced |
| HasMortgage | str | Yes / No |
| HasDependents | str | Yes / No |
| LoanPurpose | str | Home / Auto / Education / Business / Other |
| HasCoSigner | str | Yes / No |
| **Default** | int | **Target** – 1 = defaulted, 0 = not |


## 1 · Libraries

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from imblearn.over_sampling import SMOTE

## 2 · Load Raw Data

In [2]:
os.chdir("..")
df = pd.read_csv("data/raw/Loan_default.csv")
print(df.shape)
df.head()

(255347, 18)


,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


## 3 · Drop LoanID

In [3]:
# Pure identifier – no predictive value
df.drop(columns=['LoanID'], inplace=True)

## 4 · Clean Column Names

In [4]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print(df.columns.tolist())

['age', 'income', 'loanamount', 'creditscore', 'monthsemployed', 'numcreditlines', 'interestrate', 'loanterm', 'dtiratio', 'education', 'employmenttype', 'maritalstatus', 'hasmortgage', 'hasdependents', 'loanpurpose', 'hascosigner', 'default']


## 5 · Fix Data Types

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255347 entries, 0 to 255346
Data columns (total 17 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   age             255347 non-null  int64  
 1   income          255347 non-null  int64  
 2   loanamount      255347 non-null  int64  
 3   creditscore     255347 non-null  int64  
 4   monthsemployed  255347 non-null  int64  
 5   numcreditlines  255347 non-null  int64  
 6   interestrate    255347 non-null  float64
 7   loanterm        255347 non-null  int64  
 8   dtiratio        255347 non-null  float64
 9   education       255347 non-null  object 
 10  employmenttype  255347 non-null  object 
 11  maritalstatus   255347 non-null  object 
 12  hasmortgage     255347 non-null  object 
 13  hasdependents   255347 non-null  object 
 14  loanpurpose     255347 non-null  object 
 15  hascosigner     255347 non-null  object 
 16  default         255347 non-null  int64  
dtypes: float64

In [6]:
# Yes/No binary columns → 1/0
binary_cols = ['hasmortgage', 'hasdependents', 'hascosigner']
df[binary_cols] = df[binary_cols].apply(lambda col: col.map({'Yes': 1, 'No': 0}))

## 6 · Check Missing Values

In [7]:
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values")

No missing values


In [8]:
# This dataset has no missing values.
# Code below runs only if missing values appear (e.g. after merging other sources).
num_cols = df.select_dtypes(include='number').columns
cat_cols = df.select_dtypes(include='object').columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

C:\Users\User\AppData\Local\Temp\ipykernel_21540\2315995745.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mode()[0], inplace=True)


## 7 · Remove Duplicates

In [9]:
print(f"Duplicates: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

Duplicates: 0


## 8 · Treat Outliers

In [10]:
# Detect using Isolation Forest AND LOF (take intersection = more confident)
num_cols = df.select_dtypes(include='number').columns.drop('default', errors='ignore')
num_data   = df[num_cols]

iso_mask = IsolationForest(contamination=0.05, random_state=42).fit_predict(num_data) == -1
lof_mask = LocalOutlierFactor(n_neighbors=30, contamination=0.05).fit_predict(num_data) == -1

outlier_mask    = iso_mask & lof_mask
outlier_indices = df.index[outlier_mask]
n_out           = outlier_mask.sum()
threshold       = int(0.05 * len(df))

print(f"Outliers: {n_out}  |  5% threshold: {threshold}")

Outliers: 872  |  5% threshold: 12767


In [11]:
if n_out == 0:
    print("No outliers found ✅")

elif n_out < threshold:
    # Few → drop
    df.drop(index=outlier_indices, inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"Dropped {n_out} rows → new shape: {df.shape}")

else:
    # Many → clip to IQR bounds (preserves data volume)
    for col in num_cols:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        lo, hi = Q1 - 1.5*(Q3-Q1), Q3 + 1.5*(Q3-Q1)
        df.loc[outlier_indices, col] = df.loc[outlier_indices, col].clip(lo, hi)
    print(f"Clipped {n_out} outlier rows ✅")

Dropped 872 rows → new shape: (254475, 17)


## 9 · Encode Categorical Columns

In [12]:
# education has a natural order → ordinal encoding
edu_order = ['High School', "Bachelor's", "Master's", 'PhD']
df['education'] = df['education'].map({v: i for i, v in enumerate(edu_order)})

# remaining object columns → one-hot (drop_first avoids multicollinearity)
df = pd.get_dummies(df, columns=['employmenttype', 'maritalstatus', 'loanpurpose'], drop_first=True)

print(f"Shape after encoding: {df.shape}")
df.head()

Shape after encoding: (254475, 23)


,age,income,loanamount,creditscore,monthsemployed,numcreditlines,interestrate,loanterm,dtiratio,education,...,default,employmenttype_Part-time,employmenttype_Self-employed,employmenttype_Unemployed,maritalstatus_Married,maritalstatus_Single,loanpurpose_Business,loanpurpose_Education,loanpurpose_Home,loanpurpose_Other
0,56,85994,50587,520,80,4,15.23,36,0.44,1,...,0,False,False,False,False,False,False,False,False,True
1,69,50432,124440,458,15,1,4.81,60,0.68,2,...,0,False,False,False,True,False,False,False,False,True
2,46,84208,129188,451,26,3,21.17,24,0.31,2,...,1,False,False,True,False,False,False,False,False,False
3,32,31713,44799,743,0,3,7.07,24,0.23,0,...,0,False,False,False,True,False,True,False,False,False
4,60,20437,9139,633,8,4,6.51,48,0.73,1,...,0,False,False,True,False,False,False,False,False,False


## 10 · Feature Engineering

In [13]:
# ===============================
# Smart Feature Engineering
# ===============================

# 1) Loan burden relative to income
df["loan_to_income"] = df["loanamount"] / (df["income"] + 1)

# 2) Monthly income
df["monthly_income"] = df["income"] / 12

# 3) Employment stability
df["employment_ratio"] = df["monthsemployed"] / (df["age"] * 12 + 1)

# 4) Credit score groups
df["creditscore_band"] = pd.cut(df["creditscore"], bins=[0, 580, 670, 740, 850], labels=[0, 1, 2, 3]).astype(int)

# 5) High risk flag
df["high_risk_flag"] = ((df["dtiratio"] > 0.45) & (df["creditscore"] < 600)).astype(int)

print("Feature Engineering Done ")
print(df.shape)

Feature Engineering Done 
(254475, 28)


## 11 · Split X / y

In [14]:
X = df.drop(columns=['default'])
y = df['default']

print(f"X: {X.shape}  |  y: {y.shape}")
print(f"Class balance:\n{y.value_counts(normalize=True).round(3)}")

X: (254475, 27)  |  y: (254475,)
Class balance:
default
0    0.884
1    0.116
Name: proportion, dtype: float64


## 12 · Train-Test Split

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

Train: (216303, 27)  |  Test: (38172, 27)


## 13 · Handle Class Imbalance (SMOTE)

In [16]:
# SMOTE applied ONLY on training data to avoid data leakage
print("Before SMOTE:", y_train.value_counts().to_dict())

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

print("After  SMOTE:", pd.Series(y_train).value_counts().to_dict())

Before SMOTE: {0: 191212, 1: 25091}
After  SMOTE: {0: 191212, 1: 191212}


## 14 · Scaling

In [17]:
scaler  = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test  = pd.DataFrame(scaler.transform(X_test),      columns=X.columns)
print("Scaling applied")

Scaling applied


## 15 · Save Processed Data

In [18]:
os.makedirs("data/processed", exist_ok=True)

df.to_parquet("data/processed/data_processed.parquet", index=False)

X_train.to_parquet("data/processed/X_train.parquet", index=False)
X_test.to_parquet("data/processed/X_test.parquet", index=False)

pd.Series(y_train).to_frame("default").to_parquet("data/processed/y_train.parquet", index=False)

pd.Series(y_test).to_frame("default").to_parquet("data/processed/y_test.parquet", index=False)

print("Saved as Parquet ")

Saved as Parquet 
